**Link repo của tôi: https://github.com/ducphat1509-beep/artificial-intelligence**

# Báo cáo Bài tập tuần - Buổi 8
**Nguyễn Đức Phát - 24110296**

---

## 1. Vấn đề bài toán (8-Puzzle Problem)
8-Puzzle là bài toán sắp xếp 8 ô số trên bảng 3x3 chứa các số từ 1 đến 8 và một ô trống (biểu diễn bằng số 0). Mục tiêu là hoán đổi ô trống để đưa trạng thái bắt đầu (Start State) về trạng thái đích (Goal State) với số bước đi tối ưu nhất.

## 2. Ý tưởng của các thuật toán Tìm kiếm có thông tin (Informed Search)
Trong buổi học này, chúng ta triển khai và so sánh hai giải thuật tìm kiếm sử dụng hàm Heuristic ước lượng khoảng cách (ở đây sử dụng **Khoảng cách Manhattan - Manhattan Distance**):

### a. Thuật toán Tìm kiếm Tham lam (Greedy Best-First Search)
- **Hàm đánh giá**: $f(n) = h(n)$
- **Bản chất**: Thuật toán chỉ quan tâm đến việc trạng thái tiếp theo trông có vẻ gần đích nhất dựa trên ước lượng của Heuristic ($h$). Trực giác của Tham lam mang tính "cận thị" vì nó **không cộng dồn** chi phí đường đi thực tế $g(n)$ từ quá khứ, dẫn đến việc có thể chọn nhầm những đường đi vòng vèo nhưng có bước tiếp theo trông rất đẹp.

### b. Thuật toán Tìm kiếm A* (A-Star Search)
- **Hàm đánh giá**: $f(n) = g(n) + h(n)$
- **Bản chất**: Đây là sự kết hợp hoàn hảo giữa quá khứ và tương lai. Trong đó:
  - $g(n)$: Chi phí thực tế tích lũy từ Start State đến trạng thái hiện tại (mỗi bước di chuyển thông thường tính chi phí là 1).
  - $h(n)$: Chi phí ước lượng từ trạng thái hiện tại đến đích (Khoảng cách Manhattan).
- **Ưu điểm**: Khắc phục hoàn toàn nhược điểm của Tham lam, đảm bảo tìm được đường đi ngắn nhất mang tính tối ưu toàn cục nhờ sự cân bằng giữa độ dài quãng đường đã đi và khoảng cách còn lại.

---

## 3. Hướng dẫn chạy chương trình với giao diện Tùy chọn thuật toán
1. Mở file `BTVN.ipynb` trong thư mục `Buổi 8`.
2. Chạy cell code chứa chương trình Tkinter.
3. Trên thanh điều khiển bên dưới giao diện, bạn sẽ thấy thanh **Dropdown (Combobox) "Algorithm:"**.
4. Hãy chọn một trong hai thuật toán: **Greedy Search** hoặc **A* Search**.
5. Nhấp vào **Reset** hoặc chọn thuật toán để làm mới thanh trạng thái về ban đầu.
6. Sử dụng **Next Step** hoặc **Auto Run** để bắt đầu mô phỏng và so sánh trực quan các giá trị $f, g, h$ hiển thị trên từng ô trạng thái.

In [2]:
# =========================================================================
# CHƯƠNG TRÌNH GIẢI BÀI TOÁN 8-PUZZLE TÍCH HỢP GREEDY SEARCH & A* SEARCH
# - Cho phép chọn thuật toán trực tiếp từ Combobox trên giao diện Tkinter.
# - Greedy Search: f(n) = h(n) (Không cộng dồn chi phí bước đi g(n)).
# - A* Search: f(n) = g(n) + h(n) (Cân bằng chi phí quá khứ và ước lượng tương lai).
# - Hàm ước lượng h(n): Khoảng cách Manhattan (Manhattan Distance).
# - Trực quan hóa thời gian thực dữ liệu f, g, h động theo thuật toán được chọn.
# =========================================================================

import tkinter as tk
from tkinter import ttk  # Sử dụng ttk để có thanh Combobox nâng cao

START_STATE = ((1, 2, 3), (4, 0, 6), (7, 5, 8))
GOAL_STATE  = ((1, 2, 3), (4, 5, 6), (7, 8, 0))
MOVES = [
    (-1, 0, 'Up'),
    (1, 0, 'Down'),
    (0, -1, 'Left'),
    (0, 1, 'Right')
]

# ── helpers ──────────────────────────────────────────────
def get_neighbors(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                x, y = i, j
    result = []
    for dx, dy, act in MOVES:
        nx, ny = x + dx, y + dy
        if 0 <= nx < 3 and 0 <= ny < 3:
            s = [list(r) for r in state]
            s[x][y], s[nx][ny] = s[nx][ny], s[x][y]
            result.append((tuple(tuple(r) for r in s), act))
    return result

def reconstruct(node):
    path = []
    while node:
        path.append(node['state'])
        node = node['parent']
    return path[::-1]

def manhattan_distance(state, goal):
    dist = 0
    goal_pos = {}
    for r in range(3):
        for c in range(3):
            goal_pos[goal[r][c]] = (r, c)
            
    for r in range(3):
        for c in range(3):
            val = state[r][c]
            if val != 0:
                tr, tc = goal_pos[val]
                dist += abs(r - tr) + abs(c - tc)
    return dist

# ── GREEDY generator ─────────────────────────────────────
def greedy_gen(start, goal):
    start_h = manhattan_distance(start, goal)
    # Tham lam chỉ đánh giá dựa trên h(n)
    queue = [{'state': start, 'parent': None, 'h': start_h, 'f': start_h}]
    visited = set()
    expanded = 0
    generated = 0
    
    while queue:
        queue.sort(key=lambda x: x['f'])
        node = queue.pop(0)
        state = node['state']
        
        if state in visited:
            continue
        visited.add(state)
        expanded += 1
        
        frontier_nodes = queue.copy()
        yield node, frontier_nodes, list(visited), expanded, generated, None, 'Greedy'
        
        if state == goal:
            yield node, [], list(visited), expanded, generated, reconstruct(node), 'Greedy'
            return
            
        for nb, _ in get_neighbors(state):
            if nb not in visited:
                nb_h = manhattan_distance(nb, goal)
                queue.append({'state': nb, 'parent': node, 'h': nb_h, 'f': nb_h})
                generated += 1

# ── A* generator ─────────────────────────────────────────
def astar_gen(start, goal):
    start_g = 0
    start_h = manhattan_distance(start, goal)
    start_f = start_g + start_h
    queue = [{'state': start, 'parent': None, 'g': start_g, 'h': start_h, 'f': start_f}]
    visited = set()
    expanded = 0
    generated = 0
    
    while queue:
        queue.sort(key=lambda x: x['f'])
        node = queue.pop(0)
        state = node['state']
        
        if state in visited:
            continue
        visited.add(state)
        expanded += 1
        
        frontier_nodes = queue.copy()
        yield node, frontier_nodes, list(visited), expanded, generated, None, 'A*'
        
        if state == goal:
            yield node, [], list(visited), expanded, generated, reconstruct(node), 'A*'
            return
            
        for nb, _ in get_neighbors(state):
            if nb not in visited:
                nb_g = node['g'] + 1  # Chi phí đường đi tăng thêm 1 qua mỗi bước
                nb_h = manhattan_distance(nb, goal)
                nb_f = nb_g + nb_h
                queue.append({'state': nb, 'parent': node, 'g': nb_g, 'h': nb_h, 'f': nb_f})
                generated += 1

# ── Scrollable Frame ─────────────────────────────────────
class ScrollableFrame(tk.Frame):
    def __init__(self, parent, height=180):
        super().__init__(parent)
        self.h_scroll = tk.Scrollbar(self, orient='horizontal')
        self.h_scroll.pack(side='bottom', fill='x')
        self.canvas = tk.Canvas(self, height=height)
        self.canvas.pack(side='left', fill='both', expand=True)
        self.h_scroll.config(command=self.canvas.xview)
        self.canvas.configure(xscrollcommand=self.h_scroll.set)
        self.inner = tk.Frame(self.canvas)
        self.inner.bind('<Configure>', lambda e: self.canvas.configure(scrollregion=self.canvas.bbox('all')))
        self.canvas.create_window((0, 0), window=self.inner, anchor='nw')
        self.canvas.bind('<Shift-MouseWheel>', lambda e: self.canvas.xview_scroll(int(-1 * (e.delta / 120)), 'units'))

# ── GUI ──────────────────────────────────────────────────
class PuzzleGUI:
    def __init__(self, root, start, goal):
        self.root = root
        self.start = start
        self.goal = goal
        self.gen = None
        self.running = False
        self.step_count = 0
        
        root.title('8 Puzzle Informed Search – Real-time Visualization')
        sw, sh = root.winfo_screenwidth(), root.winfo_screenheight()
        w, h = min(1400, int(sw * .85)), min(900, int(sh * .85))
        root.geometry(f'{w}x{h}+{(sw-w)//2}+{(sh-h)//2}')
        root.minsize(900, 600)

        self.info = tk.Label(root, text='Press Auto Run or Next Step', font=('Arial', 14, 'bold'))
        self.info.pack(pady=6)

        # Thanh điều khiển chính bên dưới cùng
        bf = tk.Frame(root)
        bf.pack(side='bottom', fill='x', pady=8)
        
        center_f = tk.Frame(bf)
        center_f.pack(anchor='center')
        
        # Thêm Combobox chọn thuật toán
        tk.Label(center_f, text='Algorithm:', font=('Arial', 10, 'bold')).pack(side='left', padx=(5, 2))
        self.algo_combobox = ttk.Combobox(center_f, values=['Greedy Search', 'A* Search'], state='readonly', width=15)
        self.algo_combobox.set('A* Search')  # Thiết lập mặc định
        self.algo_combobox.pack(side='left', padx=5)
        self.algo_combobox.bind('<<ComboboxSelected>>', lambda e: self.reset_search())

        # Thêm nút Reset thuật toán độc lập
        tk.Button(center_f, text='Reset', width=10, command=self.reset_search, bg='#FF5722', fg='white').pack(side='left', padx=5)
        
        tk.Button(center_f, text='Next Step', width=14, command=self.next_step).pack(side='left', padx=5)
        self.auto_btn = tk.Button(center_f, text='Auto Run', width=14, command=self.toggle_auto)
        self.auto_btn.pack(side='left', padx=5)

        self.speed_var = tk.IntVar(value=300)
        tk.Label(center_f, text='Speed (ms):').pack(side='left', padx=(15, 2))
        tk.Scale(center_f, from_=10, to=1000, orient='horizontal', variable=self.speed_var, showvalue=True, length=160).pack(side='left')

        # Khu vực hiển thị đồ họa các Node
        self.cur_frame = tk.LabelFrame(root, text='Current State (Popped Node)', padx=8, pady=8)
        self.cur_frame.pack(side='top', fill='x', padx=10, pady=4)

        self.fl_frame = tk.LabelFrame(root, text='Frontier (Priority Queue)', padx=8, pady=8)
        self.fl_frame.pack(side='top', fill='both', expand=True, padx=10, pady=4)
        self.frontier_sf = ScrollableFrame(self.fl_frame, 180)
        self.frontier_sf.pack(fill='both', expand=True)

        el = tk.LabelFrame(root, text='Explored Set', padx=8, pady=8)
        el.pack(side='top', fill='both', expand=True, padx=10, pady=4)
        self.explored_sf = ScrollableFrame(el, 180)
        self.explored_sf.pack(fill='both', expand=True)
        
        # Gọi reset ban đầu để thiết lập cấu hình Generator
        self.reset_search()

    def reset_search(self):
        self.running = False
        self.step_count = 0
        self.auto_btn.config(text='Auto Run')
        self.info.config(text='Press Auto Run or Next Step')
        
        self.clear(self.cur_frame)
        self.clear(self.frontier_sf.inner)
        self.clear(self.explored_sf.inner)
        
        algo = self.algo_combobox.get()
        if algo == 'Greedy Search':
            self.gen = greedy_gen(self.start, self.goal)
            self.fl_frame.config(text='Frontier (Priority Queue - Ordered by Lowest f(n) = h(n))')
        else:
            self.gen = astar_gen(self.start, self.goal)
            self.fl_frame.config(text='Frontier (Priority Queue - Ordered by Lowest f(n) = g(n) + h(n))')
            
        # Hiển thị thông báo trạng thái ban đầu
        self.draw_state(self.cur_frame, self.start, 'Ready! Click Next Step to Start').pack()

    def draw_state(self, parent, state, label_text=None):
        container = tk.Frame(parent)
        grid_f = tk.Frame(container)
        grid_f.pack()
        for i in range(3):
            for j in range(3):
                v = state[i][j]
                tk.Label(grid_f, text=str(v) if v else '', width=4, height=2,
                         font=('Arial', 13, 'bold'), relief='solid',
                         bg='#4CAF50' if v == 0 else 'white').grid(row=i, column=j)
                         
        if label_text:
            lbl = tk.Label(container, text=label_text, font=('Arial', 10, 'bold'), fg='blue')
            lbl.pack(pady=4)
        return container

    def clear(self, frame):
        for w in frame.winfo_children(): w.destroy()

    def render(self, current_node, frontier_nodes, explored, expanded, generated, path, mode):
        self.clear(self.cur_frame)
        self.clear(self.frontier_sf.inner)
        self.clear(self.explored_sf.inner)

        # Hiển thị Current State tương ứng với từng thuật toán
        c_state = current_node['state']
        if mode == 'Greedy':
            c_h = current_node['h']
            self.draw_state(self.cur_frame, c_state, f'f = h = {c_h}').pack()
        else:
            c_f, c_g, c_h = current_node['f'], current_node['g'], current_node['h']
            self.draw_state(self.cur_frame, c_state, f'f = {c_f} (g={c_g}, h={c_h})').pack()
        
        # Hiển thị các phần tử trong Frontier theo thuật toán tương ứng
        for i, n in enumerate(frontier_nodes):
            n_state = n['state']
            if mode == 'Greedy':
                n_h = n['h']
                self.draw_state(self.frontier_sf.inner, n_state, f'f = h = {n_h}').grid(row=0, column=i, padx=8, pady=8)
            else:
                n_f, n_g, n_h = n['f'], n['g'], n['h']
                self.draw_state(self.frontier_sf.inner, n_state, f'f = {n_f} (g={n_g}, h={n_h})').grid(row=0, column=i, padx=8, pady=8)
                
        # Hiển thị Explored Set
        for i, s in enumerate(explored):
            self.draw_state(self.explored_sf.inner, s, 'Explored').grid(row=0, column=i, padx=8, pady=8)

        status = f'[{mode} Search] Step {self.step_count} | Expanded: {expanded} | Generated: {generated}'
        if path:
            status += f' | GOAL FOUND! Path length: {len(path)-1} steps'
        self.info.config(text=status)

    def next_step(self):
        try:
            current_node, frontier_nodes, explored, expanded, generated, path, mode = next(self.gen)
            self.step_count += 1
            self.render(current_node, frontier_nodes, explored, expanded, generated, path, mode)
            if path:
                self.running = False
                self.auto_btn.config(text='Auto Run')
        except StopIteration:
            self.running = False
            self.auto_btn.config(text='Done')

    def toggle_auto(self):
        self.running = not self.running
        self.auto_btn.config(text='Pause' if self.running else 'Auto Run')
        if self.running:
            self.auto_tick()

    def auto_tick(self):
        if not self.running: return
        self.next_step()
        if self.running:
            self.root.after(self.speed_var.get(), self.auto_tick)

root = tk.Tk()
app = PuzzleGUI(root, START_STATE, GOAL_STATE)
root.mainloop()